# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [8]:
df['revenue']= df['qty']*df['price']
revenue = df['revenue'].sum()

print('Total Rows:', len(df))
print('Total Revenue:', revenue)
print('Total Units:', df['qty'].sum())

Total Rows: 400
Total Revenue: 8520.0
Total Units: 783


Total Rows:400 means that there are a total of 400 orders included in the dataframe.

Total Revenue: 8520.0 means that $8,520.0 was made through the 400 transactions.

Ttoal Units: 783 means that there were 783 units, or items, sold across the 400 individual orders.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [39]:
cat_revenue= df.groupby('category')['revenue'].sum().reset_index()

cat_revenue['share_of_total'] = cat_revenue['revenue'] / df['revenue'].sum() * 100
cat_revenue['share_of_total'] = cat_revenue['share_of_total'].round(2)

cat_revenue = cat_revenue.sort_values('revenue', ascending=False)

cat_revenue

,category,revenue,share_of_total
1,Food,4293.0,50.39
2,Merch,1771.5,20.79
0,Drink,1554.0,18.24
3,RainGear,901.5,10.58


This table displays the revenue made by each category of item over the 400 orders in the dataframe in descending order. It also shows us the share of total revenue that was made by each category, rounded to 2 decimal places.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [38]:
ven_revenue= df.groupby('vendor_id').agg(avg_revenue=('revenue','mean'), orders=('qty', 'count'))

ven_revenue= ven_revenue.sort_values('avg_revenue', ascending=False).reset_index().round(2)

ven_revenue


,vendor_id,avg_revenue,orders
0,V-01,22.60,94
1,V-18,21.75,108
2,V-05,20.58,93
3,V-10,20.31,105


After creating a table from the dataframe that displays vendor ids, each one's avg revenue, and their order totals, we see that vendor V-01 has the highest average revenue (revenue per sale) at about $22.60. They ended with this average after completing 94 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [54]:
merch= df.loc[df['category'] == 'Merch', 'revenue'].sum()

share_merch= merch / df['revenue'].sum() * 100

print(f'Merch revenue share: {share_merch:.1f}%')

Merch revenue share: 20.8%


'Merch revenue share: 20.8%' tells us that the revenue made off of merch sales in the dataframe accounts for approximately 20.8% of the total revenue made throughout the 400 orders.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [78]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

base_rows= len(df)
base_revenue= df['revenue'].sum()

merged= df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one', indicator=True)

print('rows:', len(merged), '(expected', base_rows,')')
print('revenue:', merged['revenue'].sum(), '(expected', base_revenue,')')


unmatchedv= merged[merged['_merge'] == 'left_only']
print('Unmatched vendor:', unmatchedv['vendor_id'].unique())

merged['vendor_name'] = merged['vendor_name'].fillna('Unmatched Vendor')

merged[['vendor_id', 'category', 'qty', 'price', 'revenue', 'vendor_name']]


# TODO: merge, validate, and report the unmatched vendor

rows: 400 (expected 400 )
revenue: 8520.0 (expected 8520.0 )
Unmatched vendor: ['V-18']


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unmatched Vendor
2,V-18,Drink,3,4.5,13.5,Unmatched Vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unmatched Vendor
...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unmatched Vendor
396,V-01,Merch,2,24.0,48.0,Hoos Burgers
397,V-10,Food,3,7.5,22.5,Cav Merch North
398,V-18,Merch,2,24.0,48.0,Unmatched Vendor


**The unmatched vendor, and what I did about it:** _..._

The unmatched vendor was vendor V-18. This meant that V-18 was included in the original, 400 order dataframe, but was missing from the smaller vendor-names dataframe that gave us information regarding the id and names of the vendors. To remedy this issue, I discovered the unmatched vendor and opted to assign thier name as "Unmatched Vendor". I did this so that the revenue, price, and total quantity numbers would not be altered in the final report. Instead, they remain included, just with an unmatched vendor.

If we dropped all the rows that V-18 was a part of, we would have distorted revenue and sales numbers, and no one would know that it was the case. They may just think that they are the real numbers.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [91]:
pd.pivot_table(merged, index='vendor_name', columns='category', values='revenue', aggfunc='sum', fill_value=0, margins=True)
pivot.style.format("${:,.2f}")


category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,$502.50,"$1,054.50",$400.50,$175.50,"$2,133.00"
Hoos Burgers,$171.00,"$1,338.00",$373.50,$241.50,"$2,124.00"
Rotunda Tacos,$298.50,$882.00,$489.00,$244.50,"$1,914.00"
Unmatched Vendor,$582.00,"$1,018.50",$508.50,$240.00,"$2,349.00"
All,"$1,554.00","$4,293.00","$1,771.50",$901.50,"$8,520.00"


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [ ]:
# assert len(df) == 400
# assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
# assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
# assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_your answer here_